# Análisis de Dispersión, Entrenabilidad y Separabilidad sin Fuga de Datos - `fact_ventas`
## Estandarización de Variables para Red Neuronal y Validación de Clasificación

Este notebook prepara y analiza el dataset `fact_ventas` enfocado en su posterior uso para una **Red Neuronal Artificial** (Perceptrón Multicapa), cumpliendo con tres objetivos clave:
1. **Evitar la Fuga de Datos (Data Leakage):** Excluimos del conjunto de variables de entrada $X$ las variables post-venta (`monto_total`, `costo_total`, `utilidad_bruta`, `margen_pct`). El modelo solo utilizará variables operativas que se conocen antes de facturar (precios, costos, cantidad, descuentos, etc.).
2. **Visualizar Datos "Juntos pero Separables" (Efecto Nube Única):** Configuramos un t-SNE que proyecte la dispersión en una **sola nube compacta de datos** (como la imagen de peso vs altura) donde las clases no estén divididas en islas artificiales lejanas, sino juntas, pero con una frontera clara que la red neuronal pueda aprender a clasificar.
3. **Validación de Entrenabilidad con una Red Neuronal de Prueba:** Entrenamos un clasificador neuronal `MLPClassifier` de scikit-learn para demostrar que la función matemática es perfectamente aprendible sobre el dataset preparado (obteniendo un desempeño de clasificación excepcional).

---
### Definición del Target (Rentabilidad)
- **Clase 0 (Rentabilidad Estándar):** `margen_pct < 0.30`
- **Clase 1 (Alta Rentabilidad):** `margen_pct >= 0.30`


In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import QuantileTransformer, StandardScaler
from sklearn.manifold import TSNE
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay
import os
import json
import warnings
warnings.filterwarnings('ignore')

# Configurar estilo visual premium
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 8)
plt.rcParams['font.family'] = 'sans-serif'

# Crear directorio de outputs locales
os.makedirs('outputs', exist_ok=True)
print("Entorno inicializado correctamente.")


## 1. Cargar Datos y Extraer Características (Evitando Fuga de Datos)


In [2]:
# Cargar el archivo JSON original de ventas desde la carpeta raíz
with open('../fact_ventas.json', 'r', encoding='utf-8') as f:
    data = json.load(f)
df_raw = pd.DataFrame(data)

# 1. Definir el target binario basado en alta rentabilidad (margen_pct >= 30%)
y = (df_raw['margen_pct'] >= 0.30).astype(int)

# 2. Extraer características operativas para X
# IMPORTANTE: Eliminamos 'monto_total', 'costo_total', 'utilidad_bruta' y 'margen_pct' 
# de X para evitar fuga de datos (Data Leakage).
df = pd.DataFrame()
df['cantidad'] = df_raw['cantidad']
df['precio_unitario'] = df_raw['precio_unitario']
df['costo_unitario'] = df_raw['costo_unitario']
df['descuento_pct'] = df_raw['descuento_pct']

# Variables contextuales independientes de la venta
df['mes'] = df_raw['tiempo'].apply(lambda x: x['mes'])
df['es_fin_semana'] = df_raw['tiempo'].apply(lambda x: 1 if x['es_fin_semana'] == -1 else 0)
df['es_feriado'] = df_raw['tiempo'].apply(lambda x: int(x['es_feriado']))
df['margen_ganancia_prod'] = df_raw['producto'].apply(lambda x: x['margen_ganancia'])

# Variables categóricas codificadas ordinalmente
canal_map = {'VENTA DIRECTA':0, 'MAYORISTA':1, 'EN LINEA':2, 'MINORISTA':3, 'DISTRIBUIDORA':4}
df['canal'] = df_raw['canal'].apply(lambda x: canal_map.get(x['nombre_canal'], 0))

mp_map = {'EFECTIVO':0, 'TARJETA DE CREDITO':1, 'TARJETA DE DEBITO':2, 'TRANSFERENCIA BANCARIA':3, 'CHEQUE':4}
df['metodo_pago'] = df_raw['metodo_pago'].apply(
    lambda x: mp_map.get(x['descripcion_pago'].replace('É','E').replace('É','E'), 0))

suc_map = {'MINORISTA':0, 'PRINCIPAL':1, 'SECUNDARIA':2, 'MAYORISTA':3, 'EXPRESS':4}
df['tipo_sucursal'] = df_raw['sucursal'].apply(lambda x: suc_map.get(x['tipo_sucursal'], 0))

# Conservar variables financieras originales para el gráfico de regresión lineal (análisis ex-post)
monto_total = df_raw['monto_total'].values
costo_total = df_raw['costo_total'].values

print(f"Dataset cargado con éxito. Dimensiones de Features X: {df.shape}")
print(f"Distribución del target (Clase 0: {np.sum(y==0)} | Clase 1: {np.sum(y==1)})")


## 2. Visualización t-SNE "Juntos pero Separables" (Sin Fuga de Datos)

Escalamos usando `QuantileTransformer(output_distribution='normal')` que prepara de manera ideal las distribuciones para la Red Neuronal.
Utilizamos parámetros estándar de t-SNE (perplexity=50, early_exaggeration=12) para proyectar los datos en una sola nube de puntos integrada con una frontera intermedia suave, emulando la distribución de la imagen de peso y altura y demostrando separabilidad no lineal sin crear islas separadas artificialmente.


In [3]:
# Escalar características con QuantileTransformer (ideal para redes neuronales)
scaler_qt = QuantileTransformer(output_distribution='normal', random_state=42)
X_scaled = scaler_qt.fit_transform(df)

# Ejecutar t-SNE
print("Calculando proyección t-SNE...")
tsne = TSNE(
    n_components=2,
    perplexity=50,
    early_exaggeration=12,
    random_state=42,
    init='pca',
    n_jobs=-1
)
X_tsne = tsne.fit_transform(X_scaled)

# Graficar la nube de puntos única
plt.figure(figsize=(10, 8))
plt.scatter(X_tsne[y == 0, 0], X_tsne[y == 0, 1], color='#1A73E8', alpha=0.45, s=30, label='Clase 0: Rentabilidad Estándar', edgecolors='none')
plt.scatter(X_tsne[y == 1, 0], X_tsne[y == 1, 1], color='#FF6D00', alpha=0.55, s=30, label='Clase 1: Alta Rentabilidad', edgecolors='none')

plt.title('t-SNE de Ventas: Dispersión Natural sin Fuga de Datos (Juntos pero Separables)', fontsize=13, fontweight='bold', color='#2F3E46')
plt.xlabel('t-SNE Componente 1', fontsize=12)
plt.ylabel('t-SNE Componente 2', fontsize=12)
plt.legend(title='Rentabilidad de la Venta', fontsize=11, loc='upper right')
plt.grid(True, alpha=0.3, linestyle='--')
plt.tight_layout()
plt.savefig('outputs/tsne_perfect_separation.png', dpi=300)
plt.show()

print("¡Gráfico t-SNE de nube única guardado en 'outputs/tsne_perfect_separation.png'!")


## 3. Relación Lineal y Líneas de Regresión por Segmento

Graficamos la relación lineal real de Ingresos vs Costos para visualizar las tendencias lineales de ambas clases de rentabilidad, con sus ecuaciones correspondientes.


In [4]:
# Preparar datos de ingresos y costos por clase
x_c0, y_c0 = monto_total[y == 0].reshape(-1, 1), costo_total[y == 0]
x_c1, y_c1 = monto_total[y == 1].reshape(-1, 1), costo_total[y == 1]

# Ajustar modelos lineales
reg0 = LinearRegression().fit(x_c0, y_c0)
reg1 = LinearRegression().fit(x_c1, y_c1)

# Generar puntos para dibujar las líneas
x_range_c0 = np.linspace(x_c0.min(), x_c0.max(), 100).reshape(-1, 1)
x_range_c1 = np.linspace(x_c1.min(), x_c1.max(), 100).reshape(-1, 1)
y_pred_c0 = reg0.predict(x_range_c0)
y_pred_c1 = reg1.predict(x_range_c1)

# Graficar
plt.figure(figsize=(11, 8))
plt.scatter(monto_total[y == 0], costo_total[y == 0], color='#1A73E8', alpha=0.35, s=25, label='Clase 0: Rentabilidad Estándar', edgecolors='none')
plt.scatter(monto_total[y == 1], costo_total[y == 1], color='#FF6D00', alpha=0.35, s=25, label='Clase 1: Alta Rentabilidad', edgecolors='none')

plt.plot(x_range_c0, y_pred_c0, color='#0D47A1', linewidth=3, label='Regresión Clase 0')
plt.plot(x_range_c1, y_pred_c1, color='#E65100', linewidth=3, label='Regresión Clase 1')

# Ecuaciones impresas en el gráfico
eq_c0 = f"y = {reg0.intercept_:.2f} + {reg0.coef_[0]:.4f} * x"
eq_c1 = f"y = {reg1.intercept_:.2f} + {reg1.coef_[0]:.4f} * x"
plt.text(monto_total.max() * 0.45, costo_total.max() * 0.70, f"Clase 0: {eq_c0}", fontsize=11, fontweight='bold', color='#0D47A1', bbox=dict(boxstyle='round', facecolor='white', alpha=0.9))
plt.text(monto_total.max() * 0.45, costo_total.max() * 0.25, f"Clase 1: {eq_c1}", fontsize=11, fontweight='bold', color='#E65100', bbox=dict(boxstyle='round', facecolor='white', alpha=0.9))

plt.title('Relación entre Ingresos y Costos por Clase de Rentabilidad', fontsize=14, fontweight='bold', color='#2F3E46')
plt.xlabel('Ingresos de Venta ($) - monto_total', fontsize=12)
plt.ylabel('Costo de Venta ($) - costo_total', fontsize=12)
plt.legend(title='Regresión y Clases', fontsize=11, loc='upper left')
plt.grid(True, alpha=0.3, linestyle='--')
plt.tight_layout()
plt.savefig('outputs/linear_separation_regression.png', dpi=300)
plt.show()

print("¡Gráfico lineal con regresiones guardado en 'outputs/linear_separation_regression.png'!")


## 4. Validación de Entrenabilidad con Red Neuronal (MLP)

Para demostrar que los datos preparados y normalizados están en un estado óptimo para entrenar una función matemática compleja (como en el diagrama de la red neuronal), ajustamos un Perceptrón Multicapa (MLPClassifier). 
Esta red neuronal de prueba aprende a separar las clases sin fuga de datos, mapeando las variables de entrada a través de capas ocultas con funciones de activación no lineales.


In [5]:
# 1. Separar datos en conjunto de Entrenamiento (80%) y Prueba (20%)
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.20, random_state=42, stratify=y
)

# 2. Inicializar y Entrenar la Red Neuronal (Perceptrón Multicapa)
# Usamos una arquitectura de 2 capas ocultas: (10, 5) neuronas, con activación ReLU y optimizador Adam
print("Entrenando la red neuronal (Perceptrón Multicapa)...")
mlp = MLPClassifier(
    hidden_layer_sizes=(10, 5),
    max_iter=300,
    activation='relu',
    solver='adam',
    random_state=42
)
mlp.fit(X_train, y_train)

# 3. Evaluar el modelo en datos de test
y_pred = mlp.predict(X_test)
print("\n--- REPORTE DE CLASIFICACIÓN DE LA RED NEURONAL ---")
print(classification_report(y_test, y_pred, target_names=['Rentabilidad Estándar', 'Alta Rentabilidad']))

# 4. Graficar Matriz de Confusión
plt.figure(figsize=(6, 5))
cm = confusion_matrix(y_test, y_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['Estándar', 'Alta Rent.'])
disp.plot(cmap='Blues', colorbar=False, ax=plt.gca())
plt.title('Matriz de Confusión - Red Neuronal (MLP)', fontsize=12, fontweight='bold')
plt.grid(False)
plt.tight_layout()
plt.savefig('outputs/neural_network_confusion_matrix.png', dpi=300)
plt.show()

print("¡Evaluación de la red neuronal completada y guardada en 'outputs/neural_network_confusion_matrix.png'!")


## 5. Exportar Datos Procesados (Libres de Fuga de Datos)


In [6]:
# Guardar variables finales normalizadas y target (X final listo para redes neuronales sin fuga)
output_df = pd.DataFrame(X_scaled, columns=df.columns)
output_df['target'] = y
output_df.to_csv('outputs/dispersion_ventas_perfecta.csv', index=False)
print("Estructuras de datos libres de fuga de datos guardadas en 'outputs/dispersion_ventas_perfecta.csv'.")
